# AI Solution Architect Agent
Ein minimaler Agent basierend auf Google Gemini mit SQLite-Gedaechtnis.

**Abhaengigkeiten:** `google-generativeai`, `python-dotenv`, `sqlite3` (built-in), `json` (built-in)

## 1) SETUP – API Key, Gemini-Modell, SQLite-Datenbank

In [ ]:
import os
import sqlite3
import json
from datetime import datetime
from dotenv import load_dotenv
import google.generativeai as genai

# API Key aus .env laden
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    raise ValueError("GEMINI_API_KEY nicht in .env gefunden!")

# Gemini initialisieren
genai.configure(api_key=api_key)
model = genai.GenerativeModel(model_name="gemini-2.0-flash")

# SQLite-Datenbank erstellen
conn = sqlite3.connect("architect.db", check_same_thread=False)
cursor = conn.cursor()
cursor.execute("""
    CREATE TABLE IF NOT EXISTS conversations (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        role TEXT NOT NULL,
        content TEXT NOT NULL,
        timestamp TEXT NOT NULL
    )
""")
conn.commit()

print("Setup abgeschlossen. Gemini-Modell und SQLite-Datenbank bereit.")

## 2) TOOLS – Architektur-Patterns Wissensbasis & Suchfunktion

In [ ]:
# Wissensbasis: 3 Architektur-Patterns
ARCHITECTURE_PATTERNS = {
    "microservices": {
        "beschreibung": "Die Anwendung wird in kleine, unabhaengige Services aufgeteilt, die jeweils eine Geschaeftsfaehigkeit kapseln und ueber APIs kommunizieren.",
        "vorteile": [
            "Unabhaengige Deployierung einzelner Services",
            "Technologie-Freiheit pro Service",
            "Horizontale Skalierung einzelner Komponenten",
            "Fehlertoleranz durch Isolation"
        ],
        "nachteile": [
            "Hohe Komplexitaet bei verteilten Systemen",
            "Netzwerk-Latenzen zwischen Services",
            "Erschwertes Debugging und Tracing",
            "Benötigt DevOps-Reife (CI/CD, Container-Orchestrierung)"
        ],
        "use_case": "Grosse Plattformen mit vielen Teams, die unabhaengig arbeiten muessen (z.B. E-Commerce, Streaming-Dienste, SaaS-Produkte)."
    },
    "monolith": {
        "beschreibung": "Die gesamte Anwendung wird als ein einzigen deployierbaren Einheit gebaut. Alle Module teilen sich denselben Prozess und dieselbe Datenbank.",
        "vorteile": [
            "Einfache Entwicklung und Deployment",
            "Keine Netzwerk-Kommunikation zwischen Modulen",
            "Einfaches Debugging und Testing",
            "Geringe Infrastruktur-Kosten"
        ],
        "nachteile": [
            "Schwierig zu skalieren einzelner Komponenten",
            "Enge Kopplung fuehrt zu Regressionen",
            "Technologie-Wechsel nur fuer die gesamte App moeglich",
            "Deployment-Zyklen werden mit wachsendem Team langsamer"
        ],
        "use_case": "Startups, MVPs, interne Tools, kleine Teams mit begrenzten Ressourcen und klar abgegrenztem Domain-Bereich."
    },
    "event-driven": {
        "beschreibung": "Komponenten kommunizieren asynchron ueber Events. Produzenten senden Events ohne Kenntnis der Konsumenten. Ein Message-Broker (z.B. Kafka, RabbitMQ) vermittelt.",
        "vorteile": [
            "Starke Entkopplung zwischen Services",
            "Asynchrone Verarbeitung fuer bessere Performance",
            "Einfache Erweiterbarkeit durch neue Konsumenten",
            "Event-Sourcing fuer vollstaendige Audit-Trails"
        ],
        "nachteile": [
            "Eventual Consistency statt sofortiger Konsistenz",
            "Komplexe Fehlerbehandlung und Dead-Letter-Queues",
            "Schwierig zu testen und zu debuggen",
            "Benötigt zuverlaessige Message-Broker-Infrastruktur"
        ],
        "use_case": "Echtzeit-Datenverarbeitung, IoT-Plattformen, Notification-Systeme, und Anwendungen mit asynchronen Workflows (z.B. Bestellprozesse mit Lagerbestand-Updates)."
    }
}

def search_patterns(query: str) -> str:
    """Durchsucht die Wissensbasis nach passenden Architektur-Patterns."""
    query_lower = query.lower()
    results = []
    for name, pattern in ARCHITECTURE_PATTERNS.items():
        # Pruefe ob Pattern-Name oder Schluesselwoerter in der Query vorkommen
        keywords = [name] + pattern["vorteile"] + pattern["nachteile"]
        if any(kw.lower() in query_lower for kw in keywords) or name in query_lower:
            results.append({"name": name, **pattern})
    
    if not results:
        # Falls nichts passt, alle zurueckgeben
        results = [{"name": k, **v} for k, v in ARCHITECTURE_PATTERNS.items()]
    
    return json.dumps(results, indent=2, ensure_ascii=False)

# Kurztest
print("Beispielsuche 'skalierung':")
print(search_patterns("skalierung")[:200] + "...")

## 3) AGENT – System-Prompt, Chat-Historie, Gemini-Integration

In [ ]:
SYSTEM_PROMPT = """Du bist ein AI Solution Architect. Deine Aufgabe ist es, vage Business-Anforderungen zu verstehen, Rueckfragen zu stellen um Unklaerheiten zu beseitigen, und dann eine passende Architektur vorzuschlagen.

Bevor du eine Architektur empfiehlst, frage IMMER zuerst nach:
1. Cloud-Provider-Präferenz (AWS, Azure, GCP, On-Premise)
2. Budget-Rahmen
3. Skalierungsanforderungen (Nutzerzahlen, Traffic)
4. Compliance-Anforderungen (DSGVO, HIPAA, etc.)
5. Bestehende Systeme und Integrationen

Nutze das search_patterns Tool wenn du Architektur-Patterns nachschlagen musst.
Antworte immer auf Deutsch. Sei praegise und strukturiert.

Verfuegbare Architektur-Patterns in deiner Wissensbasis:
- microservices
- monolith
- event-driven

Du kannst die Funktion search_patterns(query) nutzen, um Details zu diesen Patterns abzurufen.
Die Ergebnisse werden dir als JSON-String zur Verfuegung gestellt.
"""

def save_message(role: str, content: str):
    """Speichert eine Nachricht in der SQLite-Datenbank."""
    timestamp = datetime.now().isoformat()
    cursor.execute(
        "INSERT INTO conversations (role, content, timestamp) VALUES (?, ?, ?)",
        (role, content, timestamp)
    )
    conn.commit()

def load_history() -> list:
    """Laedt die gesamte Chat-Historie aus der Datenbank."""
    cursor.execute("SELECT role, content FROM conversations ORDER BY id ASC")
    rows = cursor.fetchall()
    return [{"role": role, "parts": [content]} for role, content in rows]

def send_message(user_input: str) -> str:
    """Sendet eine Nachricht an Gemini und gibt die Antwort zurueck."""
    # 1) User-Nachricht in DB speichern
    save_message("user", user_input)
    
    # 2) Chat-Historie laden
    history = load_history()
    
    # 3) Chat-Session starten mit System-Prompt und Historie
    chat = model.start_chat(history=history)
    
    # 4) Pruefen ob wir Pattern-Suche durchfuehren sollten
    #    (simple Heuristik: wenn Architektur-relevante Begriffe auftauchen)
    architecture_keywords = ["architektur", "pattern", "microservice", "monolith", 
                            "event", "skalier", "design", "struktur", "system"]
    pattern_context = ""
    if any(kw in user_input.lower() for kw in architecture_keywords):
        pattern_context = f"\n\n[Pattern-Suche fuer '{user_input}']:\n{search_patterns(user_input)}"
    
    # 5) An Gemini senden (die eigentliche User-Nachricht + ggf. Pattern-Kontext)
    full_input = user_input + pattern_context if pattern_context else user_input
    response = chat.send_message(full_input)
    answer = response.text
    
    # 6) Antwort in DB speichern
    save_message("model", answer)
    
    return answer

print("Agent bereit. System-Prompt konfiguriert.")
print(f"Gespeicherte Nachrichten in DB: {cursor.execute('SELECT COUNT(*) FROM conversations').fetchone()[0]}")

## 4) CHAT-LOOP – Interaktives Gespraech mit dem Agent

Tippe deine Nachricht und druecke Enter. Mit `quit` beendest du den Chat.

In [ ]:
print("="*60)
print("AI Solution Architect – Chat gestartet")
print("Tippe 'quit' zum Beenden, 'history' fuer Chat-Verlauf")
print("="*60)

while True:
    user_input = input("\nDu: ").strip()
    
    if not user_input:
        continue
    
    if user_input.lower() == "quit":
        print("\nChat beendet. Auf Wiedersehen!")
        break
    
    if user_input.lower() == "history":
        print("\n--- Chat-Verlauf ---")
        cursor.execute("SELECT role, content, timestamp FROM conversations ORDER BY id ASC")
        for role, content, ts in cursor.fetchall():
            label = "Du" if role == "user" else "Architect"
            print(f"[{ts[:19]}] {label}: {content[:100]}...")
        print("--- Ende ---")
        continue
    
    try:
        response = send_message(user_input)
        print(f"\nArchitect: {response}")
    except Exception as e:
        print(f"\nFehler: {e}")

# Verbindung schliessen beim Beenden
conn.close()
print("Datenbankverbindung geschlossen.")